# March Machine Learning Mania 2026

Predict NCAA tournament outcomes (men's + women's) using an ensemble of XGBoost, LightGBM, AdaBoost, and TabICL v2.

- **Metric**: Brier score (MSE for binary outcomes)
- **Target**: P(lower TeamID beats higher TeamID)
- **Submission**: Every possible within-gender matchup

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

from theme import apply
from utils import (
    load_data, compute_elo, compute_season_stats, compute_massey_features,
    build_seed_map, build_training_data, parse_seed,
    train_or_load, brier_score, leave_one_season_out_cv,
    generate_submission,
    plot_brier_by_season, plot_feature_importance,
    plot_prediction_distribution, plot_calibration_curve,
    plot_model_comparison,
)

warnings.filterwarnings("ignore")
C = apply("mocha")
RANDOM_STATE = 42
print("Setup complete.")

## 2. Data Loading

In [ ]:
data = load_data()
for name, df in data.items():
    print(f"{name:25s} {str(df.shape):>15s}")

In [ ]:
data["m_regular_detail"].head(3)

In [ ]:
# Submission structure
print("Stage 1:", data["sample_sub"].shape)
print(data["sample_sub"].head())
print()
print("Stage 2:", data["sample_sub2"].shape)
print(data["sample_sub2"].head())

## 3. EDA

In [ ]:
# Seed vs tournament wins (men's)
m_seeds = data["m_seeds"].copy()
m_seeds["SeedNum"] = m_seeds["Seed"].apply(parse_seed)
m_tourney = data["m_tourney"]

# Count wins by seed
w_seeds = m_tourney.merge(m_seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"])
seed_wins = w_seeds.groupby("SeedNum").size().reindex(range(1, 17), fill_value=0)

l_seeds = m_tourney.merge(m_seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"])
seed_losses = l_seeds.groupby("SeedNum").size().reindex(range(1, 17), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(1, 17)
w = 0.35
ax.bar(x - w/2, seed_wins.values, w, label="Wins", color=C.green)
ax.bar(x + w/2, seed_losses.values, w, label="Losses", color=C.red)
ax.set_xlabel("Tournament Seed")
ax.set_ylabel("Count")
ax.set_title("Men's Tournament: Wins & Losses by Seed")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Upset frequency: how often does the higher seed (worse number) win?
tourney_all = pd.concat([data["m_tourney"], data["w_tourney"]], ignore_index=True)
all_seeds = pd.concat([data["m_seeds"], data["w_seeds"]], ignore_index=True)
all_seeds["SeedNum"] = all_seeds["Seed"].apply(parse_seed)
seed_lookup = dict(zip(zip(all_seeds["Season"], all_seeds["TeamID"]), all_seeds["SeedNum"]))

upsets = []
for _, row in tourney_all.iterrows():
    ws = seed_lookup.get((row["Season"], row["WTeamID"]))
    ls = seed_lookup.get((row["Season"], row["LTeamID"]))
    if ws is not None and ls is not None:
        diff = abs(ws - ls)
        upset = ws > ls  # winner had a worse (higher) seed number
        upsets.append({"SeedDiff": diff, "Upset": int(upset)})

upset_df = pd.DataFrame(upsets)
upset_rate = upset_df.groupby("SeedDiff")["Upset"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(upset_rate.index, upset_rate.values, color=C.peach, edgecolor="none")
ax.axhline(0.5, color=C.red, linestyle="--", alpha=0.5)
ax.set_xlabel("Absolute Seed Difference")
ax.set_ylabel("Upset Rate")
ax.set_title("Upset Rate by Seed Differential")
plt.tight_layout()
plt.show()

In [ ]:
# Score distributions (men's vs women's regular season)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (key, title, color) in zip(axes, [
    ("m_regular", "Men's", C.blue),
    ("w_regular", "Women's", C.pink),
]):
    df = data[key]
    scores = pd.concat([df["WScore"], df["LScore"]])
    ax.hist(scores, bins=60, color=color, edgecolor="none", alpha=0.85)
    ax.set_xlabel("Score")
    ax.set_title(f"{title} Regular Season Scores")
    ax.axvline(scores.mean(), color=C.red, linestyle="--", label=f"Mean: {scores.mean():.1f}")
    ax.legend()

axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Home court advantage
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (key, title) in zip(axes, [("m_regular", "Men's"), ("w_regular", "Women's")]):
    df = data[key]
    loc_counts = df["WLoc"].value_counts()
    total_games = len(df)
    pcts = (loc_counts / total_games * 100)
    bars = ax.bar(pcts.index, pcts.values, color=[C.green, C.blue, C.peach][:len(pcts)], edgecolor="none")
    ax.set_ylabel("% of Wins")
    ax.set_title(f"{title}: Winner's Location")
    for bar, pct in zip(bars, pcts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{pct:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Massey ordinals: systems per season
massey = data["m_massey"]
systems_per_season = massey.groupby("Season")["SystemName"].nunique()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(systems_per_season.index, systems_per_season.values, color=C.lavender, edgecolor="none")
ax.set_xlabel("Season")
ax.set_ylabel("Number of Ranking Systems")
ax.set_title("Massey Ordinals: Ranking Systems per Season")
plt.tight_layout()
plt.show()